**DNABERT + CNN + BiLSTM + Attention :**

In [1]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModel
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, roc_auc_score, matthews_corrcoef, recall_score
from collections import Counter
from tqdm import tqdm
import random, re, os, gc, time
from torch.cuda.amp import autocast, GradScaler
import seaborn as sns

# --- Reproducibility ---
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed()

# --- Utilities ---
def clean_sequence(seq):
    return re.sub('[^ATCG]', '', seq.upper())

def load_fasta_clean(path):
    with open(path) as f:
        lines = f.readlines()
    return [clean_sequence(lines[i+1].strip()) for i in range(0, len(lines), 2)]

def reverse_complement(seq):
    return seq.translate(str.maketrans("ATCG", "TAGC"))[::-1]

def kmer_tokenize(seq, k=6):
    return " ".join([seq[i:i+k] for i in range(len(seq)-k+1)])

def random_mutation(seq, mutation_rate=0.05):
    bases = ['A','T','C','G']
    seq_list = list(seq)
    n_mut = max(1, int(len(seq)*mutation_rate))
    idxs = random.sample(range(len(seq)), n_mut)
    for i in idxs:
        orig = seq_list[i]
        choices = [b for b in bases if b != orig]
        seq_list[i] = random.choice(choices)
    return "".join(seq_list)

def kmer_shuffle(seq, k=6):
    kmers = [seq[i:i+k] for i in range(len(seq)-k+1)]
    random.shuffle(kmers)
    return " ".join(kmers)

def augment_sequence(seq):
    seq = random_mutation(seq, mutation_rate=0.05)
    return kmer_shuffle(seq)

# --- Dataset ---
class SimpleDataset(Dataset):
    def __init__(self, sequences, labels, tokenizer, max_len=512, augment=False):
        self.sequences = sequences
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.augment = augment

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        seq = self.sequences[idx]
        label = self.labels[idx]
        if self.augment:
            if random.random() < 0.5:
                seq = augment_sequence(seq)
            else:
                seq = kmer_tokenize(seq)
        else:
            seq = kmer_tokenize(seq)
        encoding = self.tokenizer(seq, padding="max_length", truncation=True, max_length=self.max_len, return_tensors="pt")
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'label': torch.tensor(label, dtype=torch.float)
        }

# --- Models ---
class FeedForwardAttention(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.W = nn.Linear(input_dim, 1, bias=False)

    def forward(self, x):
        e = self.W(x).squeeze(-1)
        alpha = torch.softmax(e, dim=1).unsqueeze(-1)
        context = (alpha * x).sum(dim=1)
        return context

class Stage2Model(nn.Module):
    def __init__(self, bert_model_name="zhihan1996/DNA_bert_6", hidden_dim=128, lstm_layers=1, dropout=0.3):
        super().__init__()
        self.bert = AutoModel.from_pretrained(bert_model_name)
        self.cnn = nn.Sequential(
            nn.Conv1d(self.bert.config.hidden_size, 64, 5, padding=2),
            nn.ReLU(),
            nn.Conv1d(64, 64, 5, padding=2),
            nn.ReLU(),
            nn.MaxPool1d(2)
        )
        self.lstm = nn.LSTM(self.bert.config.hidden_size, hidden_dim, lstm_layers, batch_first=True, bidirectional=True)
        self.attention = FeedForwardAttention(hidden_dim * 2)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(64 + hidden_dim * 2, 1)

    def forward(self, input_ids, attention_mask):
        bert_outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden_states = bert_outputs.last_hidden_state
        cnn_input = last_hidden_states.permute(0, 2, 1)
        cnn_features = self.cnn(cnn_input)
        cnn_features = torch.max(cnn_features, dim=2)[0]
        lstm_out, _ = self.lstm(last_hidden_states)
        attn_out = self.attention(lstm_out)
        combined = torch.cat([cnn_features, attn_out], dim=1)
        combined = self.dropout(combined)
        logits = self.classifier(combined).squeeze(-1)
        return logits

class SimpleDNABERTClassifier(nn.Module):
    def __init__(self, bert_model_name="zhihan1996/DNA_bert_6", dropout=0.3):
        super().__init__()
        self.bert = AutoModel.from_pretrained(bert_model_name)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.bert.config.hidden_size, 1)

    def forward(self, input_ids, attention_mask):
        pooled_output = self.bert(input_ids=input_ids, attention_mask=attention_mask).pooler_output
        logits = self.classifier(self.dropout(pooled_output)).squeeze(-1)
        return logits

# --- Evaluation & Plotting ---
def compute_metrics(y_true, y_pred, y_prob):
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    return {
        'SN': recall_score(y_true, y_pred),
        'SP': tn / (tn + fp),
        'ACC': accuracy_score(y_true, y_pred),
        'F1': f1_score(y_true, y_pred),
        'MCC': matthews_corrcoef(y_true, y_pred),
        'AUC': roc_auc_score(y_true, y_prob),
        'CM': cm
    }

def plot_auc_curve(y_true, y_prob, stage, fold):
    from sklearn.metrics import roc_curve
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    plt.figure()
    plt.plot(fpr, tpr, label=f'ROC curve (AUC = {roc_auc_score(y_true, y_prob):.4f})')
    plt.plot([0, 1], [0, 1], 'k--')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'{stage} - Fold {fold+1} ROC Curve')
    plt.legend(loc='lower right')
    plt.savefig(f'{stage}_fold{fold+1}_roc.png')
    plt.close()

def plot_loss_curve(losses, stage, fold):
    plt.figure()
    plt.plot(losses, label='Training Loss')
    plt.title(f'{stage} - Fold {fold+1} Loss Curve')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.savefig(f'{stage}_fold{fold+1}_loss.png')
    plt.close()

def plot_confusion_matrix(cm, stage, fold):
    plt.figure()
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Pred 0', 'Pred 1'], yticklabels=['True 0', 'True 1'])
    plt.title(f'{stage} - Fold {fold+1} Confusion Matrix')
    plt.savefig(f'{stage}_fold{fold+1}_cm.png')
    plt.close()

# --- Data Prep ---
def prepare_stage_data(stage):
    if stage == "Stage1":
        enhancers = load_fasta_clean("/kaggle/input/taha-apu/enhancer.cv.txt")
        non_enhancers = load_fasta_clean("/kaggle/input/taha-apu/non.cv.txt")
        enhancers = enhancers[:len(non_enhancers)]
        X = enhancers + [reverse_complement(seq) for seq in enhancers] + \
            non_enhancers + [reverse_complement(seq) for seq in non_enhancers]
        y = [1]*len(enhancers)*2 + [0]*len(non_enhancers)*2
    else:
        strong = load_fasta_clean("/kaggle/input/taha-apu/strong_742.txt")
        weak = load_fasta_clean("/kaggle/input/taha-apu/weak_742.txt")
        max_len = max(len(strong), len(weak))
        strong = (strong * ((max_len // len(strong)) + 1))[:max_len]
        weak = (weak * ((max_len // len(weak)) + 1))[:max_len]
        X = strong + [reverse_complement(seq) for seq in strong] + \
            weak + [reverse_complement(seq) for seq in weak]
        y = [1]*len(strong)*2 + [0]*len(weak)*2
    return X, y

# --- Training ---
def run_cross_validation(X, y, stage, tokenizer, epochs=5, k=10):
    skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    results = []

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        print(f"\n🔁 {stage} - Fold {fold+1}/{k}")
        train_dataset = SimpleDataset([X[i] for i in train_idx], [y[i] for i in train_idx], tokenizer, augment=(stage=="Stage2"))
        val_dataset = SimpleDataset([X[i] for i in val_idx], [y[i] for i in val_idx], tokenizer)

        train_loader = DataLoader(train_dataset, batch_size=16 if stage=="Stage2" else 32, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=32)

        model = Stage2Model() if stage == "Stage2" else SimpleDNABERTClassifier()
        model = nn.DataParallel(model)
        model = model.to(device)

        optimizer = AdamW(model.parameters(), lr=1e-5 if stage == "Stage2" else 2e-5)
        loss_fn = nn.BCEWithLogitsLoss()
        scaler = GradScaler()

        losses = []
        start_time = time.time()
        torch.cuda.reset_peak_memory_stats()

        for epoch in range(epochs):
            model.train()
            total_loss = 0
            for batch in train_loader:
                optimizer.zero_grad()
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['label'].to(device)

                with autocast():
                    logits = model(input_ids, attention_mask)
                    loss = loss_fn(logits, labels)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
                total_loss += loss.item()

            avg_loss = total_loss / len(train_loader)
            losses.append(avg_loss)
            print(f"Epoch {epoch+1} Loss: {avg_loss:.4f}")

        train_time = time.time() - start_time
        memory_used = torch.cuda.max_memory_allocated() / 1e6
        param_count = sum(p.numel() for p in model.parameters() if p.requires_grad)

        model.eval()
        y_true, y_pred, y_prob = [], [], []
        inference_start = time.time()
        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['label'].to(device)
                with autocast():
                    logits = model(input_ids, attention_mask)
                    probs = torch.sigmoid(logits)
                y_true.extend(labels.cpu().tolist())
                y_prob.extend(probs.cpu().tolist())
                y_pred.extend((probs > 0.5).int().cpu().tolist())
        inference_time = time.time() - inference_start

        metrics = compute_metrics(y_true, y_pred, y_prob)
        metrics.update({
            'TrainTimeSec': train_time,
            'InferTimeSec': inference_time,
            'Params': param_count,
            'MemMB': memory_used
        })
        print(metrics)

        plot_auc_curve(y_true, y_prob, stage, fold)
        plot_loss_curve(losses, stage, fold)
        plot_confusion_matrix(metrics['CM'], stage, fold)
        results.append(metrics)

    avg = {k: np.mean([fold[k] for fold in results if k in fold]) for k in results[0]}
    print(f"\n=== Final {stage} {k}-Fold Averages ===")
    for k, v in avg.items():
        print(f"{k}: {v:.4f}")

# --- Run ---
if __name__ == "__main__":
    tokenizer = AutoTokenizer.from_pretrained("zhihan1996/DNA_bert_6", do_lower_case=False, trust_remote_code=True)
    for stage in ["Stage1", "Stage2"]:
        X, y = prepare_stage_data(stage)
        run_cross_validation(X, y, stage, tokenizer, epochs=5, k=10)


tokenizer_config.json:   0%|          | 0.00/40.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.45k [00:00<?, ?B/s]

configuration_bert.py:   0%|          | 0.00/807 [00:00<?, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/zhihan1996/DNA_bert_6:
- configuration_bert.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


vocab.txt:   0%|          | 0.00/28.7k [00:00<?, ?B/s]

...

{'SN': 0.7520215633423181, 'SP': 0.40431266846361186, 'ACC': 0.578167115902965, 'F1': 0.6406429391504019, 'MCC': 0.16673827662706, 'AUC': 0.6118707361905247, 'CM': array([[150, 221],
       [ 92, 279]]), 'TrainTimeSec': 672.8026535511017, 'InferTimeSec': 4.112538576126099, 'Params': 90377409, 'MemMB': 3123.444224}

🔁 Stage2 - Fold 10/10


/tmp/ipykernel_35/1144574374.py:227: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
/tmp/ipykernel_35/1144574374.py:242: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch 1 Loss: 0.6899
Epoch 2 Loss: 0.6828
Epoch 3 Loss: 0.6824
Epoch 4 Loss: 0.6791
Epoch 5 Loss: 0.6734


/tmp/ipykernel_35/1144574374.py:266: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


{'SN': 0.41509433962264153, 'SP': 0.7493261455525606, 'ACC': 0.5822102425876011, 'F1': 0.4983818770226538, 'MCC': 0.17445312720552855, 'AUC': 0.6139994623694974, 'CM': array([[278,  93],
       [217, 154]]), 'TrainTimeSec': 671.9950745105743, 'InferTimeSec': 4.098861455917358, 'Params': 90377409, 'MemMB': 3128.136192}

=== Final Stage2 10-Fold Averages ===
SN: 0.5720
SP: 0.5507
ACC: 0.5613
F1: 0.5488
MCC: 0.1330
AUC: 0.5972
CM: 185.5000
TrainTimeSec: 673.2258
InferTimeSec: 4.1596
Params: 90377409.0000
MemMB: 3129.6106


**DNABERT ONLY:**

In [1]:
# DNA BERT Stage 2 Variant 1: DNABERT Only (Full Pipeline with 10-Fold Evaluation and Reporting)

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModel
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, roc_auc_score, matthews_corrcoef, recall_score
from collections import Counter
from tqdm import tqdm
import random, re, os, gc, time
from torch.cuda.amp import autocast, GradScaler
import seaborn as sns

# --- Reproducibility ---
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed()

# --- Utilities ---
def clean_sequence(seq):
    return re.sub('[^ATCG]', '', seq.upper())

def load_fasta_clean(path):
    with open(path) as f:
        lines = f.readlines()
    return [clean_sequence(lines[i+1].strip()) for i in range(0, len(lines), 2)]

def reverse_complement(seq):
    return seq.translate(str.maketrans("ATCG", "TAGC"))[::-1]

def kmer_tokenize(seq, k=6):
    return " ".join([seq[i:i+k] for i in range(len(seq)-k+1)])

def random_mutation(seq, mutation_rate=0.05):
    bases = ['A','T','C','G']
    seq_list = list(seq)
    n_mut = max(1, int(len(seq)*mutation_rate))
    idxs = random.sample(range(len(seq)), n_mut)
    for i in idxs:
        orig = seq_list[i]
        choices = [b for b in bases if b != orig]
        seq_list[i] = random.choice(choices)
    return "".join(seq_list)

def kmer_shuffle(seq, k=6):
    kmers = [seq[i:i+k] for i in range(len(seq)-k+1)]
    random.shuffle(kmers)
    return " ".join(kmers)

def augment_sequence(seq):
    seq = random_mutation(seq, mutation_rate=0.05)
    return kmer_shuffle(seq)

# --- Dataset ---
class SimpleDataset(Dataset):
    def __init__(self, sequences, labels, tokenizer, max_len=512, augment=False):
        self.sequences = sequences
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.augment = augment

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        seq = self.sequences[idx]
        label = self.labels[idx]
        if self.augment:
            if random.random() < 0.5:
                seq = augment_sequence(seq)
            else:
                seq = kmer_tokenize(seq)
        else:
            seq = kmer_tokenize(seq)
        encoding = self.tokenizer(seq, padding="max_length", truncation=True, max_length=self.max_len, return_tensors="pt")
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'label': torch.tensor(label, dtype=torch.float)
        }

# --- Stage 2 Variant 1: DNABERT Only ---
class Stage2Model(nn.Module):
    def __init__(self, bert_model_name="zhihan1996/DNA_bert_6", dropout=0.3):
        super().__init__()
        self.bert = AutoModel.from_pretrained(bert_model_name)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.bert.config.hidden_size, 1)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.pooler_output  # [batch_size, hidden_dim]
        logits = self.classifier(self.dropout(pooled_output)).squeeze(-1)
        return logits

# --- Evaluation & Plotting ---
def compute_metrics(y_true, y_pred, y_prob):
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    return {
        'SN': recall_score(y_true, y_pred),
        'SP': tn / (tn + fp),
        'ACC': accuracy_score(y_true, y_pred),
        'F1': f1_score(y_true, y_pred),
        'MCC': matthews_corrcoef(y_true, y_pred),
        'AUC': roc_auc_score(y_true, y_prob),
        'CM': cm
    }

def plot_auc_curve(y_true, y_prob, stage, fold):
    from sklearn.metrics import roc_curve
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    plt.figure()
    plt.plot(fpr, tpr, label=f'ROC curve (AUC = {roc_auc_score(y_true, y_prob):.4f})')
    plt.plot([0, 1], [0, 1], 'k--')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'{stage} - Fold {fold+1} ROC Curve')
    plt.legend(loc='lower right')
    plt.savefig(f'{stage}_fold{fold+1}_roc.png')
    plt.close()

def plot_loss_curve(losses, stage, fold):
    plt.figure()
    plt.plot(losses, label='Training Loss')
    plt.title(f'{stage} - Fold {fold+1} Loss Curve')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.savefig(f'{stage}_fold{fold+1}_loss.png')
    plt.close()

def plot_confusion_matrix(cm, stage, fold):
    plt.figure()
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Pred 0', 'Pred 1'], yticklabels=['True 0', 'True 1'])
    plt.title(f'{stage} - Fold {fold+1} Confusion Matrix')
    plt.savefig(f'{stage}_fold{fold+1}_cm.png')
    plt.close()

# --- Data Prep ---
def prepare_stage_data(stage):
    strong = load_fasta_clean("/kaggle/input/taha-apu/strong_742.txt")
    weak = load_fasta_clean("/kaggle/input/taha-apu/weak_742.txt")
    max_len = max(len(strong), len(weak))
    strong = (strong * ((max_len // len(strong)) + 1))[:max_len]
    weak = (weak * ((max_len // len(weak)) + 1))[:max_len]
    X = strong + [reverse_complement(seq) for seq in strong] + \
        weak + [reverse_complement(seq) for seq in weak]
    y = [1]*len(strong)*2 + [0]*len(weak)*2
    return X, y

# --- Training ---
def run_cross_validation(X, y, stage, tokenizer, epochs=5, k=10):
    skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    results = []

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        print(f"\n🔁 {stage} - Fold {fold+1}/{k}")
        train_dataset = SimpleDataset([X[i] for i in train_idx], [y[i] for i in train_idx], tokenizer, augment=True)
        val_dataset = SimpleDataset([X[i] for i in val_idx], [y[i] for i in val_idx], tokenizer)

        train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=32)

        model = Stage2Model()
        model = nn.DataParallel(model)
        model = model.to(device)

        optimizer = AdamW(model.parameters(), lr=1e-5)
        loss_fn = nn.BCEWithLogitsLoss()
        scaler = GradScaler()

        losses = []
        start_time = time.time()
        torch.cuda.reset_peak_memory_stats()

        for epoch in range(epochs):
            model.train()
            total_loss = 0
            for batch in train_loader:
                optimizer.zero_grad()
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['label'].to(device)

                with autocast():
                    logits = model(input_ids, attention_mask)
                    loss = loss_fn(logits, labels)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
                total_loss += loss.item()

            avg_loss = total_loss / len(train_loader)
            losses.append(avg_loss)
            print(f"Epoch {epoch+1} Loss: {avg_loss:.4f}")

        train_time = time.time() - start_time
        memory_used = torch.cuda.max_memory_allocated() / 1e6
        param_count = sum(p.numel() for p in model.parameters() if p.requires_grad)

        model.eval()
        y_true, y_pred, y_prob = [], [], []
        inference_start = time.time()
        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['label'].to(device)
                with autocast():
                    logits = model(input_ids, attention_mask)
                    probs = torch.sigmoid(logits)
                y_true.extend(labels.cpu().tolist())
                y_prob.extend(probs.cpu().tolist())
                y_pred.extend((probs > 0.5).int().cpu().tolist())
        inference_time = time.time() - inference_start

        metrics = compute_metrics(y_true, y_pred, y_prob)
        metrics.update({
            'TrainTimeSec': train_time,
            'InferTimeSec': inference_time,
            'Params': param_count,
            'MemMB': memory_used
        })
        print(metrics)

        plot_auc_curve(y_true, y_prob, stage, fold)
        plot_loss_curve(losses, stage, fold)
        plot_confusion_matrix(metrics['CM'], stage, fold)
        results.append(metrics)

    avg = {k: np.mean([fold[k] for fold in results if k in fold]) for k in results[0]}
    print(f"\n=== Final {stage} {k}-Fold Averages ===")
    for k, v in avg.items():
        print(f"{k}: {v:.4f}")

# --- Run ---
if __name__ == "__main__":
    tokenizer = AutoTokenizer.from_pretrained("zhihan1996/DNA_bert_6", do_lower_case=False, trust_remote_code=True)
    X, y = prepare_stage_data("Stage2")
    run_cross_validation(X, y, "Stage2", tokenizer, epochs=5, k=10)


tokenizer_config.json:   0%|          | 0.00/40.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.45k [00:00<?, ?B/s]

configuration_bert.py:   0%|          | 0.00/807 [00:00<?, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/zhihan1996/DNA_bert_6:
- configuration_bert.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


vocab.txt:   0%|          | 0.00/28.7k [00:00<?, ?B/s]

...

{'SN': 0.862533692722372, 'SP': 0.2830188679245283, 'ACC': 0.5727762803234502, 'F1': 0.6687565308254964, 'MCC': 0.17860049270284542, 'AUC': 0.6021207343742054, 'CM': array([[105, 266],
       [ 51, 320]]), 'TrainTimeSec': 634.1851167678833, 'InferTimeSec': 3.9231624603271484, 'Params': 89191681, 'MemMB': 3070.571008}

🔁 Stage2 - Fold 10/10


/tmp/ipykernel_35/1839221622.py:182: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
/tmp/ipykernel_35/1839221622.py:197: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch 1 Loss: 0.6929
Epoch 2 Loss: 0.6859
Epoch 3 Loss: 0.6849
Epoch 4 Loss: 0.6818
Epoch 5 Loss: 0.6795


/tmp/ipykernel_35/1839221622.py:221: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


{'SN': 0.3638814016172507, 'SP': 0.7789757412398922, 'ACC': 0.5714285714285714, 'F1': 0.45918367346938777, 'MCC': 0.1570241137734644, 'AUC': 0.6206326603265016, 'CM': array([[289,  82],
       [236, 135]]), 'TrainTimeSec': 635.0265944004059, 'InferTimeSec': 3.9550137519836426, 'Params': 89191681, 'MemMB': 3074.578944}

=== Final Stage2 10-Fold Averages ===
SN: 0.6569
SP: 0.4461
ACC: 0.5515
F1: 0.5729
MCC: 0.1185
AUC: 0.5874
CM: 185.5000
TrainTimeSec: 633.6959
InferTimeSec: 3.9499
Params: 89191681.0000
MemMB: 3071.3449


**DNABERT + CNN :**

In [1]:
# DNA BERT Stage 2 Variant 2: DNABERT + CNN (Full Pipeline with 10-Fold Evaluation and Reporting)

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModel
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, roc_auc_score, matthews_corrcoef, recall_score
from collections import Counter
from tqdm import tqdm
import random, re, os, gc, time
from torch.cuda.amp import autocast, GradScaler
import seaborn as sns

# --- Reproducibility ---
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed()

# --- Utilities ---
def clean_sequence(seq):
    return re.sub('[^ATCG]', '', seq.upper())

def load_fasta_clean(path):
    with open(path) as f:
        lines = f.readlines()
    return [clean_sequence(lines[i+1].strip()) for i in range(0, len(lines), 2)]

def reverse_complement(seq):
    return seq.translate(str.maketrans("ATCG", "TAGC"))[::-1]

def kmer_tokenize(seq, k=6):
    return " ".join([seq[i:i+k] for i in range(len(seq)-k+1)])

def random_mutation(seq, mutation_rate=0.05):
    bases = ['A','T','C','G']
    seq_list = list(seq)
    n_mut = max(1, int(len(seq)*mutation_rate))
    idxs = random.sample(range(len(seq)), n_mut)
    for i in idxs:
        orig = seq_list[i]
        choices = [b for b in bases if b != orig]
        seq_list[i] = random.choice(choices)
    return "".join(seq_list)

def kmer_shuffle(seq, k=6):
    kmers = [seq[i:i+k] for i in range(len(seq)-k+1)]
    random.shuffle(kmers)
    return " ".join(kmers)

def augment_sequence(seq):
    seq = random_mutation(seq, mutation_rate=0.05)
    return kmer_shuffle(seq)

# --- Dataset ---
class SimpleDataset(Dataset):
    def __init__(self, sequences, labels, tokenizer, max_len=512, augment=False):
        self.sequences = sequences
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.augment = augment

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        seq = self.sequences[idx]
        label = self.labels[idx]
        if self.augment:
            if random.random() < 0.5:
                seq = augment_sequence(seq)
            else:
                seq = kmer_tokenize(seq)
        else:
            seq = kmer_tokenize(seq)
        encoding = self.tokenizer(seq, padding="max_length", truncation=True, max_length=self.max_len, return_tensors="pt")
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'label': torch.tensor(label, dtype=torch.float)
        }

# --- Stage 2 Variant 2: DNABERT + CNN ---
class Stage2Model(nn.Module):
    def __init__(self, bert_model_name="zhihan1996/DNA_bert_6", dropout=0.3):
        super().__init__()
        self.bert = AutoModel.from_pretrained(bert_model_name)
        self.cnn = nn.Sequential(
            nn.Conv1d(self.bert.config.hidden_size, 64, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2)
        )
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(64, 1)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden = outputs.last_hidden_state
        cnn_input = last_hidden.permute(0, 2, 1)
        cnn_features = self.cnn(cnn_input)
        pooled = torch.max(cnn_features, dim=2)[0]
        logits = self.classifier(self.dropout(pooled)).squeeze(-1)
        return logits

# --- Evaluation & Plotting ---
def compute_metrics(y_true, y_pred, y_prob):
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    return {
        'SN': recall_score(y_true, y_pred),
        'SP': tn / (tn + fp),
        'ACC': accuracy_score(y_true, y_pred),
        'F1': f1_score(y_true, y_pred),
        'MCC': matthews_corrcoef(y_true, y_pred),
        'AUC': roc_auc_score(y_true, y_prob),
        'CM': cm
    }

def plot_auc_curve(y_true, y_prob, stage, fold):
    from sklearn.metrics import roc_curve
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    plt.figure()
    plt.plot(fpr, tpr, label=f'ROC curve (AUC = {roc_auc_score(y_true, y_prob):.4f})')
    plt.plot([0, 1], [0, 1], 'k--')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'{stage} - Fold {fold+1} ROC Curve')
    plt.legend(loc='lower right')
    plt.savefig(f'{stage}_fold{fold+1}_roc.png')
    plt.close()

def plot_loss_curve(losses, stage, fold):
    plt.figure()
    plt.plot(losses, label='Training Loss')
    plt.title(f'{stage} - Fold {fold+1} Loss Curve')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.savefig(f'{stage}_fold{fold+1}_loss.png')
    plt.close()

def plot_confusion_matrix(cm, stage, fold):
    plt.figure()
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Pred 0', 'Pred 1'], yticklabels=['True 0', 'True 1'])
    plt.title(f'{stage} - Fold {fold+1} Confusion Matrix')
    plt.savefig(f'{stage}_fold{fold+1}_cm.png')
    plt.close()

# --- Data Prep ---
def prepare_stage_data(stage):
    strong = load_fasta_clean("/kaggle/input/taha-apu/strong_742.txt")
    weak = load_fasta_clean("/kaggle/input/taha-apu/weak_742.txt")
    max_len = max(len(strong), len(weak))
    strong = (strong * ((max_len // len(strong)) + 1))[:max_len]
    weak = (weak * ((max_len // len(weak)) + 1))[:max_len]
    X = strong + [reverse_complement(seq) for seq in strong] + \
        weak + [reverse_complement(seq) for seq in weak]
    y = [1]*len(strong)*2 + [0]*len(weak)*2
    return X, y

# --- Training ---
def run_cross_validation(X, y, stage, tokenizer, epochs=5, k=10):
    skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    results = []

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        print(f"\n🔁 {stage} - Fold {fold+1}/{k}")
        train_dataset = SimpleDataset([X[i] for i in train_idx], [y[i] for i in train_idx], tokenizer, augment=True)
        val_dataset = SimpleDataset([X[i] for i in val_idx], [y[i] for i in val_idx], tokenizer)

        train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=32)

        model = Stage2Model()
        model = nn.DataParallel(model)
        model = model.to(device)

        optimizer = AdamW(model.parameters(), lr=1e-5)
        loss_fn = nn.BCEWithLogitsLoss()
        scaler = GradScaler()

        losses = []
        start_time = time.time()
        torch.cuda.reset_peak_memory_stats()

        for epoch in range(epochs):
            model.train()
            total_loss = 0
            for batch in train_loader:
                optimizer.zero_grad()
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['label'].to(device)

                with autocast():
                    logits = model(input_ids, attention_mask)
                    loss = loss_fn(logits, labels)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
                total_loss += loss.item()

            avg_loss = total_loss / len(train_loader)
            losses.append(avg_loss)
            print(f"Epoch {epoch+1} Loss: {avg_loss:.4f}")

        train_time = time.time() - start_time
        memory_used = torch.cuda.max_memory_allocated() / 1e6
        param_count = sum(p.numel() for p in model.parameters() if p.requires_grad)

        model.eval()
        y_true, y_pred, y_prob = [], [], []
        inference_start = time.time()
        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['label'].to(device)
                with autocast():
                    logits = model(input_ids, attention_mask)
                    probs = torch.sigmoid(logits)
                y_true.extend(labels.cpu().tolist())
                y_prob.extend(probs.cpu().tolist())
                y_pred.extend((probs > 0.5).int().cpu().tolist())
        inference_time = time.time() - inference_start

        metrics = compute_metrics(y_true, y_pred, y_prob)
        metrics.update({
            'TrainTimeSec': train_time,
            'InferTimeSec': inference_time,
            'Params': param_count,
            'MemMB': memory_used
        })
        print(metrics)

        plot_auc_curve(y_true, y_prob, stage, fold)
        plot_loss_curve(losses, stage, fold)
        plot_confusion_matrix(metrics['CM'], stage, fold)
        results.append(metrics)

    avg = {k: np.mean([fold[k] for fold in results if k in fold]) for k in results[0]}
    print(f"\n=== Final {stage} {k}-Fold Averages ===")
    for k, v in avg.items():
        print(f"{k}: {v:.4f}")

# --- Run ---
if __name__ == "__main__":
    tokenizer = AutoTokenizer.from_pretrained("zhihan1996/DNA_bert_6", do_lower_case=False, trust_remote_code=True)
    X, y = prepare_stage_data("Stage2")
    run_cross_validation(X, y, "Stage2", tokenizer, epochs=5, k=10)


tokenizer_config.json:   0%|          | 0.00/40.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.45k [00:00<?, ?B/s]

configuration_bert.py:   0%|          | 0.00/807 [00:00<?, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/zhihan1996/DNA_bert_6:
- configuration_bert.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


vocab.txt:   0%|          | 0.00/28.7k [00:00<?, ?B/s]

...

{'SN': 0.24258760107816713, 'SP': 0.8544474393530997, 'ACC': 0.5485175202156334, 'F1': 0.34951456310679613, 'MCC': 0.12267912301642961, 'AUC': 0.6085468719349613, 'CM': array([[317,  54],
       [281,  90]]), 'TrainTimeSec': 665.918249130249, 'InferTimeSec': 4.088600397109985, 'Params': 89436801, 'MemMB': 3075.645952}

🔁 Stage2 - Fold 10/10


/tmp/ipykernel_35/3552925903.py:190: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
/tmp/ipykernel_35/3552925903.py:205: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch 1 Loss: 0.7086
Epoch 2 Loss: 0.6902
Epoch 3 Loss: 0.6817
Epoch 4 Loss: 0.6791
Epoch 5 Loss: 0.6761


/tmp/ipykernel_35/3552925903.py:229: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


{'SN': 0.4420485175202156, 'SP': 0.7304582210242587, 'ACC': 0.5862533692722371, 'F1': 0.5165354330708661, 'MCC': 0.18016236003427666, 'AUC': 0.619800786102978, 'CM': array([[271, 100],
       [207, 164]]), 'TrainTimeSec': 664.3604271411896, 'InferTimeSec': 4.105581521987915, 'Params': 89436801, 'MemMB': 3074.778624}

=== Final Stage2 10-Fold Averages ===
SN: 0.5596
SP: 0.5566
ACC: 0.5581
F1: 0.5303
MCC: 0.1334
AUC: 0.5960
CM: 185.5000
TrainTimeSec: 664.9711
InferTimeSec: 4.1479
Params: 89436801.0000
MemMB: 3074.3784


**DNABERT + BiLSTM**

In [1]:
# DNA BERT Stage 2 Variant 3: DNABERT + BiLSTM (Full Pipeline with 10-Fold Evaluation and Reporting)

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModel
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, roc_auc_score, matthews_corrcoef, recall_score
from collections import Counter
from tqdm import tqdm
import random, re, os, gc, time
from torch.cuda.amp import autocast, GradScaler
import seaborn as sns

# --- Reproducibility ---
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed()

# --- Utilities ---
def clean_sequence(seq):
    return re.sub('[^ATCG]', '', seq.upper())

def load_fasta_clean(path):
    with open(path) as f:
        lines = f.readlines()
    return [clean_sequence(lines[i+1].strip()) for i in range(0, len(lines), 2)]

def reverse_complement(seq):
    return seq.translate(str.maketrans("ATCG", "TAGC"))[::-1]

def kmer_tokenize(seq, k=6):
    return " ".join([seq[i:i+k] for i in range(len(seq)-k+1)])

def random_mutation(seq, mutation_rate=0.05):
    bases = ['A','T','C','G']
    seq_list = list(seq)
    n_mut = max(1, int(len(seq)*mutation_rate))
    idxs = random.sample(range(len(seq)), n_mut)
    for i in idxs:
        orig = seq_list[i]
        choices = [b for b in bases if b != orig]
        seq_list[i] = random.choice(choices)
    return "".join(seq_list)

def kmer_shuffle(seq, k=6):
    kmers = [seq[i:i+k] for i in range(len(seq)-k+1)]
    random.shuffle(kmers)
    return " ".join(kmers)

def augment_sequence(seq):
    seq = random_mutation(seq, mutation_rate=0.05)
    return kmer_shuffle(seq)

# --- Dataset ---
class SimpleDataset(Dataset):
    def __init__(self, sequences, labels, tokenizer, max_len=512, augment=False):
        self.sequences = sequences
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.augment = augment

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        seq = self.sequences[idx]
        label = self.labels[idx]
        if self.augment:
            if random.random() < 0.5:
                seq = augment_sequence(seq)
            else:
                seq = kmer_tokenize(seq)
        else:
            seq = kmer_tokenize(seq)
        encoding = self.tokenizer(seq, padding="max_length", truncation=True, max_length=self.max_len, return_tensors="pt")
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'label': torch.tensor(label, dtype=torch.float)
        }

# --- Stage 2 Variant 3: DNABERT + BiLSTM ---
class Stage2Model(nn.Module):
    def __init__(self, bert_model_name="zhihan1996/DNA_bert_6", hidden_dim=128, dropout=0.3):
        super().__init__()
        self.bert = AutoModel.from_pretrained(bert_model_name)
        self.lstm = nn.LSTM(self.bert.config.hidden_size, hidden_dim, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_dim * 2, 1)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden = outputs.last_hidden_state
        lstm_out, _ = self.lstm(last_hidden)
        pooled = torch.mean(lstm_out, dim=1)
        logits = self.classifier(self.dropout(pooled)).squeeze(-1)
        return logits

# --- Evaluation ---
def compute_metrics(y_true, y_pred, y_prob):
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    return {
        'SN': recall_score(y_true, y_pred),
        'SP': tn / (tn + fp),
        'ACC': accuracy_score(y_true, y_pred),
        'F1': f1_score(y_true, y_pred),
        'MCC': matthews_corrcoef(y_true, y_pred),
        'AUC': roc_auc_score(y_true, y_prob),
        'CM': cm
    }

def plot_auc_curve(y_true, y_prob, stage, fold):
    from sklearn.metrics import roc_curve
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    plt.figure()
    plt.plot(fpr, tpr, label=f'ROC curve (AUC = {roc_auc_score(y_true, y_prob):.4f})')
    plt.plot([0, 1], [0, 1], 'k--')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'{stage} - Fold {fold+1} ROC Curve')
    plt.legend(loc='lower right')
    plt.savefig(f'{stage}_fold{fold+1}_roc.png')
    plt.close()

def plot_loss_curve(losses, stage, fold):
    plt.figure()
    plt.plot(losses, label='Training Loss')
    plt.title(f'{stage} - Fold {fold+1} Loss Curve')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.savefig(f'{stage}_fold{fold+1}_loss.png')
    plt.close()

def plot_confusion_matrix(cm, stage, fold):
    plt.figure()
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Pred 0', 'Pred 1'], yticklabels=['True 0', 'True 1'])
    plt.title(f'{stage} - Fold {fold+1} Confusion Matrix')
    plt.savefig(f'{stage}_fold{fold+1}_cm.png')
    plt.close()

# --- Data Prep ---
def prepare_stage_data(stage):
    strong = load_fasta_clean("/kaggle/input/taha-apu/strong_742.txt")
    weak = load_fasta_clean("/kaggle/input/taha-apu/weak_742.txt")
    max_len = max(len(strong), len(weak))
    strong = (strong * ((max_len // len(strong)) + 1))[:max_len]
    weak = (weak * ((max_len // len(weak)) + 1))[:max_len]
    X = strong + [reverse_complement(seq) for seq in strong] + \
        weak + [reverse_complement(seq) for seq in weak]
    y = [1]*len(strong)*2 + [0]*len(weak)*2
    return X, y

# --- Training ---
def run_cross_validation(X, y, stage, tokenizer, epochs=5, k=10):
    skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    results = []

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        print(f"\n🔁 {stage} - Fold {fold+1}/{k}")
        train_dataset = SimpleDataset([X[i] for i in train_idx], [y[i] for i in train_idx], tokenizer, augment=True)
        val_dataset = SimpleDataset([X[i] for i in val_idx], [y[i] for i in val_idx], tokenizer)

        train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=32)

        model = Stage2Model()
        model = nn.DataParallel(model)
        model = model.to(device)

        optimizer = AdamW(model.parameters(), lr=1e-5)
        loss_fn = nn.BCEWithLogitsLoss()
        scaler = GradScaler()

        losses = []
        start_time = time.time()
        torch.cuda.reset_peak_memory_stats()

        for epoch in range(epochs):
            model.train()
            total_loss = 0
            for batch in train_loader:
                optimizer.zero_grad()
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['label'].to(device)

                with autocast():
                    logits = model(input_ids, attention_mask)
                    loss = loss_fn(logits, labels)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
                total_loss += loss.item()

            avg_loss = total_loss / len(train_loader)
            losses.append(avg_loss)
            print(f"Epoch {epoch+1} Loss: {avg_loss:.4f}")

        train_time = time.time() - start_time
        memory_used = torch.cuda.max_memory_allocated() / 1e6
        param_count = sum(p.numel() for p in model.parameters() if p.requires_grad)

        model.eval()
        y_true, y_pred, y_prob = [], [], []
        inference_start = time.time()
        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['label'].to(device)
                with autocast():
                    logits = model(input_ids, attention_mask)
                    probs = torch.sigmoid(logits)
                y_true.extend(labels.cpu().tolist())
                y_prob.extend(probs.cpu().tolist())
                y_pred.extend((probs > 0.5).int().cpu().tolist())
        inference_time = time.time() - inference_start

        metrics = compute_metrics(y_true, y_pred, y_prob)
        metrics.update({
            'TrainTimeSec': train_time,
            'InferTimeSec': inference_time,
            'Params': param_count,
            'MemMB': memory_used
        })
        print(metrics)

        plot_auc_curve(y_true, y_prob, stage, fold)
        plot_loss_curve(losses, stage, fold)
        plot_confusion_matrix(metrics['CM'], stage, fold)
        results.append(metrics)

    avg = {k: np.mean([fold[k] for fold in results if k in fold]) for k in results[0]}
    print(f"\n=== Final {stage} {k}-Fold Averages ===")
    for k, v in avg.items():
        print(f"{k}: {v:.4f}")

# --- Run ---
if __name__ == "__main__":
    tokenizer = AutoTokenizer.from_pretrained("zhihan1996/DNA_bert_6", do_lower_case=False, trust_remote_code=True)
    X, y = prepare_stage_data("Stage2")
    run_cross_validation(X, y, "Stage2", tokenizer, epochs=5, k=10)


tokenizer_config.json:   0%|          | 0.00/40.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.45k [00:00<?, ?B/s]

configuration_bert.py:   0%|          | 0.00/807 [00:00<?, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/zhihan1996/DNA_bert_6:
- configuration_bert.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


vocab.txt:   0%|          | 0.00/28.7k [00:00<?, ?B/s]

...

{'SN': 0.816711590296496, 'SP': 0.32075471698113206, 'ACC': 0.568733153638814, 'F1': 0.6544276457883369, 'MCC': 0.15830800694034733, 'AUC': 0.6059168416387558, 'CM': array([[119, 252],
       [ 68, 303]]), 'TrainTimeSec': 658.9275000095367, 'InferTimeSec': 4.020500183105469, 'Params': 90110721, 'MemMB': 3141.059584}

🔁 Stage2 - Fold 10/10


/tmp/ipykernel_35/2393777006.py:185: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
/tmp/ipykernel_35/2393777006.py:200: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch 1 Loss: 0.6896
Epoch 2 Loss: 0.6829
Epoch 3 Loss: 0.6792
Epoch 4 Loss: 0.6796
Epoch 5 Loss: 0.6745


/tmp/ipykernel_35/2393777006.py:224: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


{'SN': 0.3908355795148248, 'SP': 0.7762803234501348, 'ACC': 0.5835579514824798, 'F1': 0.48414023372287146, 'MCC': 0.1811101150675472, 'AUC': 0.6143917873308099, 'CM': array([[288,  83],
       [226, 145]]), 'TrainTimeSec': 657.7551553249359, 'InferTimeSec': 3.9964537620544434, 'Params': 90110721, 'MemMB': 3142.147072}

=== Final Stage2 10-Fold Averages ===
SN: 0.5709
SP: 0.5418
ACC: 0.5563
F1: 0.5464
MCC: 0.1226
AUC: 0.5916
CM: 185.5000
TrainTimeSec: 658.0439
InferTimeSec: 4.0833
Params: 90110721.0000
MemMB: 3140.7574


**DNABERT + Transformer Encoder :**

In [3]:
# DNA BERT Stage 2 Variant 4: DNABERT + Transformer Encoder (Full Pipeline with 10-Fold Evaluation and Reporting)

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModel
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, roc_auc_score, matthews_corrcoef, recall_score
from collections import Counter
from tqdm import tqdm
import random, re, os, gc, time
from torch.cuda.amp import autocast, GradScaler
import seaborn as sns

# --- Reproducibility 
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed()

# --- Utilities ---
def clean_sequence(seq):
    return re.sub('[^ATCG]', '', seq.upper())

def load_fasta_clean(path):
    with open(path) as f:
        lines = f.readlines()
    return [clean_sequence(lines[i+1].strip()) for i in range(0, len(lines), 2)]

def reverse_complement(seq):
    return seq.translate(str.maketrans("ATCG", "TAGC"))[::-1]

def kmer_tokenize(seq, k=6):
    return " ".join([seq[i:i+k] for i in range(len(seq)-k+1)])

def random_mutation(seq, mutation_rate=0.05):
    bases = ['A','T','C','G']
    seq_list = list(seq)
    n_mut = max(1, int(len(seq)*mutation_rate))
    idxs = random.sample(range(len(seq)), n_mut)
    for i in idxs:
        orig = seq_list[i]
        choices = [b for b in bases if b != orig]
        seq_list[i] = random.choice(choices)
    return "".join(seq_list)

def kmer_shuffle(seq, k=6):
    kmers = [seq[i:i+k] for i in range(len(seq)-k+1)]
    random.shuffle(kmers)
    return " ".join(kmers)

def augment_sequence(seq):
    seq = random_mutation(seq, mutation_rate=0.05)
    return kmer_shuffle(seq)

# --- Dataset ---
class SimpleDataset(Dataset):
    def __init__(self, sequences, labels, tokenizer, max_len=512, augment=False):
        self.sequences = sequences
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.augment = augment

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        seq = self.sequences[idx]
        label = self.labels[idx]
        if self.augment:
            if random.random() < 0.5:
                seq = augment_sequence(seq)
            else:
                seq = kmer_tokenize(seq)
        else:
            seq = kmer_tokenize(seq)
        encoding = self.tokenizer(seq, padding="max_length", truncation=True, max_length=self.max_len, return_tensors="pt")
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'label': torch.tensor(label, dtype=torch.float)
        }

# --- Stage 2 Variant 4: DNABERT + Transformer Encoder ---
class TransformerEncoderBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, dropout=0.1):
        super().__init__()
        self.attn = nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout, batch_first=True)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.ff = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 4),
            nn.ReLU(),
            nn.Linear(embed_dim * 4, embed_dim)
        )
        self.norm2 = nn.LayerNorm(embed_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        attn_output, _ = self.attn(x, x, x)
        x = self.norm1(x + self.dropout(attn_output))
        ff_output = self.ff(x)
        x = self.norm2(x + self.dropout(ff_output))
        return x

class Stage2Model(nn.Module):
    def __init__(self, bert_model_name="zhihan1996/DNA_bert_6", dropout=0.3):
        super().__init__()
        self.bert = AutoModel.from_pretrained(bert_model_name)
        self.transformer = TransformerEncoderBlock(self.bert.config.hidden_size, num_heads=4, dropout=dropout)
        self.pooling = nn.AdaptiveAvgPool1d(1)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.bert.config.hidden_size, 1)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        hidden_states = outputs.last_hidden_state
        encoded = self.transformer(hidden_states)
        pooled = self.pooling(encoded.transpose(1, 2)).squeeze(-1)
        logits = self.classifier(self.dropout(pooled)).squeeze(-1)
        return logits

# --- Evaluation & Plotting ---
def compute_metrics(y_true, y_pred, y_prob):
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    return {
        'SN': recall_score(y_true, y_pred),
        'SP': tn / (tn + fp),
        'ACC': accuracy_score(y_true, y_pred),
        'F1': f1_score(y_true, y_pred),
        'MCC': matthews_corrcoef(y_true, y_pred),
        'AUC': roc_auc_score(y_true, y_prob),
        'CM': cm
    }

def plot_auc_curve(y_true, y_prob, stage, fold):
    from sklearn.metrics import roc_curve
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    plt.figure()
    plt.plot(fpr, tpr, label=f'ROC curve (AUC = {roc_auc_score(y_true, y_prob):.4f})')
    plt.plot([0, 1], [0, 1], 'k--')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'{stage} - Fold {fold+1} ROC Curve')
    plt.legend(loc='lower right')
    plt.savefig(f'{stage}_fold{fold+1}_roc.png')
    plt.close()

def plot_loss_curve(losses, stage, fold):
    plt.figure()
    plt.plot(losses, label='Training Loss')
    plt.title(f'{stage} - Fold {fold+1} Loss Curve')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.savefig(f'{stage}_fold{fold+1}_loss.png')
    plt.close()

def plot_confusion_matrix(cm, stage, fold):
    plt.figure()
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Pred 0', 'Pred 1'], yticklabels=['True 0', 'True 1'])
    plt.title(f'{stage} - Fold {fold+1} Confusion Matrix')
    plt.savefig(f'{stage}_fold{fold+1}_cm.png')
    plt.close()

# --- Data Prep ---
def prepare_stage_data(stage):
    strong = load_fasta_clean("/kaggle/input/taha-apu/strong_742.txt")
    weak = load_fasta_clean("/kaggle/input/taha-apu/weak_742.txt")
    max_len = max(len(strong), len(weak))
    strong = (strong * ((max_len // len(strong)) + 1))[:max_len]
    weak = (weak * ((max_len // len(weak)) + 1))[:max_len]
    X = strong + [reverse_complement(seq) for seq in strong] + \
        weak + [reverse_complement(seq) for seq in weak]
    y = [1]*len(strong)*2 + [0]*len(weak)*2
    return X, y

# --- Training ---
def run_cross_validation(X, y, stage, tokenizer, epochs=5, k=10):
    skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    results = []

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        print(f"\n🔁 {stage} - Fold {fold+1}/{k}")
        train_dataset = SimpleDataset([X[i] for i in train_idx], [y[i] for i in train_idx], tokenizer, augment=True)
        val_dataset = SimpleDataset([X[i] for i in val_idx], [y[i] for i in val_idx], tokenizer)

        train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=32)

        model = Stage2Model()
        model = nn.DataParallel(model)
        model = model.to(device)

        optimizer = AdamW(model.parameters(), lr=1e-5)
        loss_fn = nn.BCEWithLogitsLoss()
        scaler = GradScaler()

        losses = []
        start_time = time.time()
        torch.cuda.reset_peak_memory_stats()

        for epoch in range(epochs):
            model.train()
            total_loss = 0
            for batch in train_loader:
                optimizer.zero_grad()
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['label'].to(device)

                with autocast():
                    logits = model(input_ids, attention_mask)
                    loss = loss_fn(logits, labels)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
                total_loss += loss.item()

            avg_loss = total_loss / len(train_loader)
            losses.append(avg_loss)
            print(f"Epoch {epoch+1} Loss: {avg_loss:.4f}")

        train_time = time.time() - start_time
        memory_used = torch.cuda.max_memory_allocated() / 1e6
        param_count = sum(p.numel() for p in model.parameters() if p.requires_grad)

        model.eval()
        y_true, y_pred, y_prob = [], [], []
        inference_start = time.time()
        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['label'].to(device)
                with autocast():
                    logits = model(input_ids, attention_mask)
                    probs = torch.sigmoid(logits)
                y_true.extend(labels.cpu().tolist())
                y_prob.extend(probs.cpu().tolist())
                y_pred.extend((probs > 0.5).int().cpu().tolist())
        inference_time = time.time() - inference_start

        metrics = compute_metrics(y_true, y_pred, y_prob)
        metrics.update({
            'TrainTimeSec': train_time,
            'InferTimeSec': inference_time,
            'Params': param_count,
            'MemMB': memory_used
        })
        print(metrics)

        plot_auc_curve(y_true, y_prob, stage, fold)
        plot_loss_curve(losses, stage, fold)
        plot_confusion_matrix(metrics['CM'], stage, fold)
        results.append(metrics)

    avg = {k: np.mean([fold[k] for fold in results if k in fold]) for k in results[0]}
    print(f"\n=== Final {stage} {k}-Fold Averages ===")
    for k, v in avg.items():
        print(f"{k}: {v:.4f}")

# --- Run ---
if __name__ == "__main__":
    tokenizer = AutoTokenizer.from_pretrained("zhihan1996/DNA_bert_6", do_lower_case=False, trust_remote_code=True)
    X, y = prepare_stage_data("Stage2")
    run_cross_validation(X, y, "Stage2", tokenizer, epochs=5, k=10)


tokenizer_config.json:   0%|          | 0.00/40.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.45k [00:00<?, ?B/s]

configuration_bert.py:   0%|          | 0.00/807 [00:00<?, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/zhihan1996/DNA_bert_6:
- configuration_bert.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


vocab.txt:   0%|          | 0.00/28.7k [00:00<?, ?B/s]

...

{'SN': 0.7304582210242587, 'SP': 0.4123989218328841, 'ACC': 0.5714285714285714, 'F1': 0.6302325581395348, 'MCC': 0.15068193182178116, 'AUC': 0.5932570963593696, 'CM': array([[153, 218],
       [100, 271]]), 'TrainTimeSec': 748.1105954647064, 'InferTimeSec': 4.582813501358032, 'Params': 96279553, 'MemMB': 3328.156672}

🔁 Stage2 - Fold 10/10


/tmp/ipykernel_35/1513702424.py:206: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
/tmp/ipykernel_35/1513702424.py:221: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch 1 Loss: 0.6971
Epoch 2 Loss: 0.6897
Epoch 3 Loss: 0.6832
Epoch 4 Loss: 0.6790
Epoch 5 Loss: 0.6732


/tmp/ipykernel_35/1513702424.py:245: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


{'SN': 0.477088948787062, 'SP': 0.6981132075471698, 'ACC': 0.5876010781671159, 'F1': 0.5363636363636364, 'MCC': 0.17964508278100633, 'AUC': 0.6319083703257025, 'CM': array([[259, 112],
       [194, 177]]), 'TrainTimeSec': 747.2478671073914, 'InferTimeSec': 4.585822105407715, 'Params': 96279553, 'MemMB': 3329.449984}

=== Final Stage2 10-Fold Averages ===
SN: 0.4965
SP: 0.6275
ACC: 0.5620
F1: 0.5159
MCC: 0.1334
AUC: 0.6006
CM: 185.5000
TrainTimeSec: 746.4209
InferTimeSec: 4.6050
Params: 96279553.0000
MemMB: 3327.6238
